# A tiny GPT, trained from scratch

**Track:** Training LLMs. **Level:** Intermediate. **CPU:** yes.

## Goals

- Read a decoder-only Transformer: token embedding, position embedding, causal self-attention, MLP block, tied loss.
- Train it for a few dozen steps on a repeated sentence.
- Sample from the trained model and check that the loss moved.

## Sources and license

Original implementation of the nanoGPT layout (Karpathy, MIT): pre-norm residual blocks, causal multi-head attention, and a next-token cross-entropy. nanoGPT and llm.c are not copied. The corpus is original and repetitive so a CPU run of well under a minute can reduce the loss. Tiny Shakespeare, the usual nanoGPT demo corpus, is public domain. It is not vendored here.

## Concepts

The model defines a distribution over the next character given the previous characters, inside a window of `block_size`. The causal mask is what makes that distribution not depend on future characters. The loss is the mean cross-entropy of those next-character predictions. Generation draws from the softmax of the last position and appends the draw. A drop in loss on this corpus means the network assigned more probability to the characters that actually repeat. It does not mean the network has learned English.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 'torch', 'numpy', 'matplotlib'])
print("packages ready")


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "labs" / "_shared" / "llm.py").exists():
            return cand
    if "google.colab" in sys.modules:
        root = Path("/content/ShaiLearnsAI")
        marker = root / "labs" / "_shared" / "llm.py"
        if not marker.exists():
            os.system(
                "git clone --depth 1 https://github.com/shai200/ShaiLearnsAI.git /content/ShaiLearnsAI"
            )
        return root
    raise RuntimeError(
        "Clone https://github.com/shai200/ShaiLearnsAI and run this notebook from that directory."
    )

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
print("repo", ROOT)

%matplotlib inline
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

from labs._shared.seed import seed_everything

seed_everything(0)
torch.manual_seed(0)

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        if n_embd % n_head != 0:
            raise ValueError("n_embd must be divisible by n_head")
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer("mask", mask)

    def forward(self, x):
        b, t, c = x.shape
        qkv = self.qkv(x).view(b, t, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.mask[:, :, :t, :t] == 0, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        mixed = (weights @ v).transpose(1, 2).contiguous().view(b, t, c)
        return self.proj(mixed)


class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class TinyGPT(nn.Module):
    """A nanoGPT-style decoder. Original code, same block layout as nanoGPT (MIT)."""

    def __init__(self, vocab_size, n_embd=64, n_head=4, n_layer=2, block_size=32):
        super().__init__()
        self.block_size = block_size
        self.tok = nn.Embedding(vocab_size, n_embd)
        self.pos = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList(
            [Block(n_embd, n_head, block_size) for _ in range(n_layer)]
        )
        self.ln = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx, targets=None, loss_mask=None):
        b, t = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(t, device=idx.device))
        for block in self.blocks:
            x = block(x)
        logits = self.head(self.ln(x))
        loss = None
        if targets is not None:
            per = nn.functional.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
                reduction="none",
            ).view(b, t)
            if loss_mask is None:
                loss = per.mean()
            else:
                loss = (per * loss_mask).sum() / loss_mask.sum().clamp_min(1.0)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n_new, temperature=0.8):
        self.eval()
        for _ in range(n_new):
            logits, _ = self(idx[:, -self.block_size :])
            probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            nxt = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


def train_lm(data, vocab_size, steps, n_embd=64, n_layer=2, n_head=4, block_size=32, batch_size=32, lr=3e-3, seed=0):
    g = torch.Generator().manual_seed(seed)
    torch.manual_seed(seed)
    model = TinyGPT(vocab_size, n_embd=n_embd, n_head=n_head, n_layer=n_layer, block_size=block_size)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []
    for _ in range(steps):
        ix = torch.randint(0, len(data) - block_size - 1, (batch_size,), generator=g)
        x = torch.stack([data[i : i + block_size] for i in ix])
        y = torch.stack([data[i + 1 : i + 1 + block_size] for i in ix])
        _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        losses.append(float(loss.detach()))
    return model, losses


## Guided exercise: overfit a repeated pattern

The vocabulary is the set of characters in the text. Training uses random windows. The assertion compares the mean loss of the first five steps with the mean of the last five.


In [ ]:
text = ("the cat sat on the mat. the cat sat on the hat. the dog sat on the log. " * 50)
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)
print("chars", len(text), "vocab", len(chars))
model, losses = train_lm(
    data, vocab_size=len(chars), steps=80, n_embd=64, n_layer=2, n_head=4, block_size=32, batch_size=32, lr=3e-3,
)
print("loss start", round(float(np.mean(losses[:5])), 3), "loss end", round(float(np.mean(losses[-5:])), 3))
assert np.mean(losses[-5:]) < 0.75 * np.mean(losses[:5])
start = torch.tensor([[stoi["t"]]])
sample = model.generate(start, n_new=60, temperature=0.7)[0].tolist()
print("".join(itos[i] for i in sample))

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(losses)
ax.set_xlabel("step")
ax.set_ylabel("cross-entropy")
ax.set_title("Tiny GPT on a repeated sentence")
plt.tight_layout()
plt.show()
print("LAB OK")


## Your turn

Count parameters. For this configuration the count is fixed. Print it and check that the token embedding and the output head are separate tensors. nanoGPT often ties them. This lab does not, so you can see the two shapes.

## Solution


In [ ]:
n_params = sum(p.numel() for p in model.parameters())
print("parameters", n_params)
print("token embedding", tuple(model.tok.weight.shape), "lm head", tuple(model.head.weight.shape))
assert model.tok.weight.data_ptr() != model.head.weight.data_ptr()
assert n_params > 10_000


## Beyond the course

Train the same data for the same number of steps with one layer and with two layers. Both should reduce the loss. The comparison is which one finishes lower at this step budget, not which architecture is better in general. A deeper model has more compute per step, so a lower loss is not a free win.


In [ ]:
_, one = train_lm(data, len(chars), steps=40, n_embd=64, n_layer=1, n_head=4, block_size=32, seed=1)
_, two = train_lm(data, len(chars), steps=40, n_embd=64, n_layer=2, n_head=4, block_size=32, seed=1)
print("1 layer", round(float(np.mean(one[-5:])), 3), "2 layers", round(float(np.mean(two[-5:])), 3))
assert np.mean(one[-5:]) < np.mean(one[:5])
assert np.mean(two[-5:]) < np.mean(two[:5])


## Research questions

1. The sample is still mostly the training sentence. Which failure would tell you the model is copying the window rather than estimating a next-character distribution?
2. Temperature 0.7 was used for the sample and 1 is implicit in the loss. Why can the loss fall while a high-temperature sample still looks random?
3. What would you have to log, besides the scalar loss, before claiming the attention heads specialized?

## Further reading

- Karpathy, nanoGPT (MIT). https://github.com/karpathy/nanoGPT. Layout reference, not copied.
- Karpathy, llm.c (MIT), the same model in C.
- Vaswani et al., *Attention Is All You Need* (2017).
- Radford et al., GPT-2 technical report (2019).
